# DSA 4060 – Practical Quiz: Personalized Movie Recommender

**Name:** Abdirisak Hussein
**Student ID:** 668776  
**Assigned User ID:** 37


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


STUDENT_ID = "668776"

last_two = int("".join(ch for ch in STUDENT_ID if ch.isdigit())[-2:])
ASSIGNED_USER_ID = (last_two % 40) + 1
print(f"Student ID: {STUDENT_ID} | last two digits: {last_two} | "
      f"{last_two} MOD 40 = {last_two % 40} | Assigned User ID = {ASSIGNED_USER_ID}")

Student ID: 668776 | last two digits: 76 | 76 MOD 40 = 36 | Assigned User ID = 37


## Task 1 – Load and Inspect the Data

In [2]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

print("movies.csv – first five rows")
display(movies.head())
print("ratings.csv – first five rows")
display(ratings.head())

movies.csv – first five rows


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


ratings.csv – first five rows


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


In [3]:
print(f"movies : {movies.shape[0]} rows x {movies.shape[1]} columns")
print(f"ratings: {ratings.shape[0]} rows x {ratings.shape[1]} columns")

movies : 36 rows x 3 columns
ratings: 440 rows x 3 columns


In [4]:
print("Missing values in movies:\n", movies.isnull().sum(), "\n")
print("Missing values in ratings:\n", ratings.isnull().sum())

Missing values in movies:
 movie_id    0
title       0
genres      0
dtype: int64 

Missing values in ratings:
 user_id     0
movie_id    0
rating      0
dtype: int64


**Finding:** *In the Movies and ratings dataset we can see that there is no missing values*

## Task 2 – Profile the Assigned User

In [5]:
user_ratings = (ratings[ratings["user_id"] == ASSIGNED_USER_ID]
               .merge(movies, on="movie_id")
               [["movie_id", "title", "genres", "rating"]]
               .sort_values("rating", ascending=False)
               .reset_index(drop=True))
print(f"User {ASSIGNED_USER_ID} has rated {len(user_ratings)} movies")
display(user_ratings)

User 37 has rated 11 movies


,movie_id,title,genres,rating
0,36,Remember the Titans,Sports|Drama,5.0
1,33,Ford v Ferrari,Sports|Drama,5.0
2,17,The Pursuit of Happyness,Drama|Biography,5.0
3,13,Titanic,Romance|Drama,4.0
4,19,The Social Network,Drama|Biography,4.0
5,34,Creed,Sports|Drama,3.5
6,14,The Notebook,Romance|Drama,3.5
7,35,Moneyball,Sports|Drama,3.5
8,18,The Shawshank Redemption,Drama,3.5
9,21,Toy Story,Animation|Comedy,3.0


In [26]:
df_37 = ratings[ratings["user_id"] == 37]
df_37.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11 entries, 395 to 405
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   user_id   11 non-null     int64  
 1   movie_id  11 non-null     int64  
 2   rating    11 non-null     float64
dtypes: float64(1), int64(2)
memory usage: 352.0 bytes


In [6]:
top3 = user_ratings.head(3)[["title", "genres", "rating"]]
print("Three highest-rated movies:")
display(top3)

Three highest-rated movies:


,title,genres,rating
0,Remember the Titans,Sports|Drama,5.0
1,Ford v Ferrari,Sports|Drama,5.0
2,The Pursuit of Happyness,Drama|Biography,5.0


In [7]:

genre_stats = (user_ratings.assign(genre=user_ratings["genres"].str.split("|"))
               .explode("genre")
               .groupby("genre")["rating"]
               .agg(n_movies="count", avg_rating="mean")
               .sort_values(["avg_rating", "n_movies"], ascending=False)
               .round(2))
display(genre_stats)

,n_movies,avg_rating
genre,,
Biography,2,4.50
Sports,4,4.25
Drama,9,4.11
Romance,3,3.00
Animation,1,3.00
Comedy,2,2.25


**Preferred genre(s):** *Biography was the highest rated genre even though it was 2 movies, on the other side drama was the third in the list with a lot of number of movies(9) with 4.11 avg_rating and sports was the second in the list with 4.25. Comedy was the at the bottom of the list with average rating of 2.25*

## Task 3 – Build the Recommendation Logic

### 3.1 Movies the user likes (rating ≥ 4.0)

In [8]:
liked = user_ratings[user_ratings["rating"] >= 4.0]
print(f"{len(liked)} liked movies (rating >= 4.0):")
display(liked[["title", "genres", "rating"]])

5 liked movies (rating >= 4.0):


,title,genres,rating
0,Remember the Titans,Sports|Drama,5.0
1,Ford v Ferrari,Sports|Drama,5.0
2,The Pursuit of Happyness,Drama|Biography,5.0
3,Titanic,Romance|Drama,4.0
4,The Social Network,Drama|Biography,4.0


### 3.2 Convert genres to numerical features (CountVectorizer)

In [9]:
vectorizer = CountVectorizer(tokenizer=lambda s: s.split("|"), token_pattern=None, lowercase=False)
genre_matrix = vectorizer.fit_transform(movies["genres"])

print("Genre vocabulary:", list(vectorizer.get_feature_names_out()))
print("Feature matrix shape (movies x genres):", genre_matrix.shape)

Genre vocabulary: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Drama', 'Family', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sports', 'Superhero', 'Thriller']
Feature matrix shape (movies x genres): (36, 16)


### 3.3 Cosine similarity between liked movies and the rest of the catalogue

In [10]:
idx = {mid: i for i, mid in enumerate(movies["movie_id"])}
liked_idx = [idx[m] for m in liked["movie_id"]]

# similarity of every movie (rows) to every liked movie (columns)
sim = cosine_similarity(genre_matrix, genre_matrix[liked_idx])
sim_df = pd.DataFrame(sim, index=movies["movie_id"], columns=liked["movie_id"].values)

# Score = mean similarity to the user's liked movies, weighted by how much they liked each one
weights = liked["rating"].values
score = (sim_df.values * weights).sum(axis=1) / weights.sum()
scores = pd.Series(score, index=movies["movie_id"], name="score")
display(scores.sort_values(ascending=False).head(10).to_frame())

,score
movie_id,
33,0.717391
34,0.717391
35,0.717391
36,0.717391
18,0.707107
19,0.695652
20,0.695652
17,0.695652
13,0.586957


### 3.4 Exclude already-rated movies and return exactly five recommendations

In [11]:
avg_rating = ratings.groupby("movie_id")["rating"].mean().rename("community_avg")

candidates = (movies.set_index("movie_id")
              .join(scores).join(avg_rating)
              .loc[lambda d: ~d.index.isin(user_ratings["movie_id"])]
              .sort_values(["score", "community_avg"], ascending=False))

top5 = candidates.head(5).reset_index()
assert len(top5) == 5
assert not set(top5["movie_id"]) & set(user_ratings["movie_id"]), "A rated movie slipped in!"
print("Recommended titles:", top5["title"].tolist())

Recommended titles: ['Hidden Figures', 'The Grand Budapest Hotel', 'Interstellar', 'Blade Runner 2049', 'Arrival']


## Task 4 – Explain the Recommendations

In [12]:
rows = []
for rank, r in enumerate(top5.itertuples(), start=1):
    sims = sim_df.loc[r.movie_id]
    best_id = sims.idxmax()
    best = liked.set_index("movie_id").loc[best_id]
    shared = sorted(set(r.genres.split("|")) & set(best["genres"].split("|")))
    liked_genres = set("|".join(liked["genres"]).split("|"))
    overlap = sorted(set(r.genres.split("|")) & liked_genres)
    reason = (f"Most similar to '{best['title']}' (rated {best['rating']}); "
              f"shares genre(s): {', '.join(shared) if shared else 'none'}. "
              f"Matches liked genres: {', '.join(overlap) if overlap else 'none'} "
              f"(similarity score {r.score:.2f}).")
    rows.append([rank, r.title, r.genres, reason])

results = pd.DataFrame(rows, columns=["Rank", "Recommended Movie", "Genres", "Reason"])
pd.set_option("display.max_colwidth", None)
display(results)

,Rank,Recommended Movie,Genres,Reason
0,1,Hidden Figures,Drama|Biography,"Most similar to 'The Pursuit of Happyness' (rated 5.0); shares genre(s): Biography, Drama. Matches liked genres: Biography, Drama (similarity score 0.70)."
1,2,The Grand Budapest Hotel,Comedy|Drama,Most similar to 'Remember the Titans' (rated 5.0); shares genre(s): Drama. Matches liked genres: Drama (similarity score 0.50).
2,3,Interstellar,Sci-Fi|Drama,Most similar to 'Remember the Titans' (rated 5.0); shares genre(s): Drama. Matches liked genres: Drama (similarity score 0.50).
3,4,Blade Runner 2049,Sci-Fi|Drama,Most similar to 'Remember the Titans' (rated 5.0); shares genre(s): Drama. Matches liked genres: Drama (similarity score 0.50).
4,5,Arrival,Sci-Fi|Drama,Most similar to 'Remember the Titans' (rated 5.0); shares genre(s): Drama. Matches liked genres: Drama (similarity score 0.50).


From the above; The reasons are accurate, but they aren't distinct.

In [39]:
rows = []
liked_genres = set("|".join(liked["genres"]).split("|"))
liked_by_id = liked.set_index("movie_id")

for rank, r in enumerate(top5.itertuples(), start=1):
    sims = sim_df.loc[r.movie_id].sort_values(ascending=False)
    sims = sims[sims > 0]
    cand_genres = set(r.genres.split("|"))


    refs = [f"'{liked_by_id.loc[m, 'title']}' ({liked_by_id.loc[m, 'rating']})"
            for m in sims.index[:2]]
    shared = sorted(cand_genres & liked_genres)
    new = sorted(cand_genres - liked_genres)

    reason = (f"Shares {', '.join(shared)} with liked movies "
              f"{' and '.join(refs)} (score {r.score:.2f}). ")
    if new:
        reason += f"Also adds {', '.join(new)}, which is new to this user's liked list. "
    reason += f"Ranked above others with the same score by community avg rating {r.community_avg:.2f}."
    rows.append([rank, r.title, r.genres, reason])

results = pd.DataFrame(rows, columns=["Rank", "Recommended Movie", "Genres", "Reason"])
pd.set_option("display.max_colwidth", None)
display(results)

,Rank,Recommended Movie,Genres,Reason
0,1,Hidden Figures,Drama|Biography,"Shares Biography, Drama with liked movies 'The Pursuit of Happyness' (5.0) and 'The Social Network' (4.0) (score 0.70). Ranked above others with the same score by community avg rating 3.47."
1,2,The Grand Budapest Hotel,Comedy|Drama,"Shares Drama with liked movies 'Remember the Titans' (5.0) and 'Ford v Ferrari' (5.0) (score 0.50). Also adds Comedy, which is new to this user's liked list. Ranked above others with the same score by community avg rating 3.31."
2,3,Interstellar,Sci-Fi|Drama,"Shares Drama with liked movies 'Remember the Titans' (5.0) and 'Ford v Ferrari' (5.0) (score 0.50). Also adds Sci-Fi, which is new to this user's liked list. Ranked above others with the same score by community avg rating 3.24."
3,4,Blade Runner 2049,Sci-Fi|Drama,"Shares Drama with liked movies 'Remember the Titans' (5.0) and 'Ford v Ferrari' (5.0) (score 0.50). Also adds Sci-Fi, which is new to this user's liked list. Ranked above others with the same score by community avg rating 3.23."
4,5,Arrival,Sci-Fi|Drama,"Shares Drama with liked movies 'Remember the Titans' (5.0) and 'Ford v Ferrari' (5.0) (score 0.50). Also adds Sci-Fi, which is new to this user's liked list. Ranked above others with the same score by community avg rating 3.15."


## Task 5 – Critical Thinking

**Limitation:** *The recommender depends only on genre labels, so movies that share one broad genre look equally similar. I mean In my results, four of the five recommendations tied at a similarity score of 0.50 because they only shared "Drama" with Remember the Titans, even though they differ a lot in style and theme. The user's small number of ratings also makes the profile weak*

**Suggested improvement:** *I would basically add filtering, finding users with similar rating patterns and using their high ratings to rank candidates. This would use real preference data to separate movies that genre alone treats as identical, making the recommendations more personalized*

In [27]:
!pwd

/content


In [28]:
!ls

movies.csv  ratings.csv  sample_data


In [29]:
!git config --global user.name "abdiri1800"
!git config --global user.email "husseinabdirizak87@gmail.com"

In [30]:
!git config --global user.name
!git config --global user.email

abdiri1800
husseinabdirizak87@gmail.com


In [31]:
!git clone https://github.com/abdiri1800/DSA4060-Recommender-668776.git

Cloning into 'DSA4060-Recommender-668776'...


In [32]:
%cd DSA4060-Recommender-668776

/content/DSA4060-Recommender-668776


In [37]:
!pwd

/content/DSA4060-Recommender-668776


In [38]:
!ls

In [34]:
!cp /content/movies.csv
!cp /content/ratings.csv

cp: missing destination file operand after '/content/movies.csv'
Try 'cp --help' for more information.
cp: missing destination file operand after '/content/ratings.csv'
Try 'cp --help' for more information.
